# Estimateurs de covariance
**Question de recherche.** Quel estimateur de la matrice de covariance Σ est le plus pertinent
pour l'optimisation de portefeuille sur le S&P 500, évalué **exclusivement hors échantillon**
selon **deux axes** :
1. **Statistique / prédictif** 
   est-il proche de la covariance réalisée (RMSE/MAE/Frobenius), bien conditionné et stable ?
2. **Portefeuille** les allocations construites sur ce Σ sont-elles meilleures en risque
   réalisé, Sharpe net, drawdown, turnover ?

**Cadre expérimental.** Une seule variable change à la fois (`dataclasses.replace`), mêmes
données / fenêtres / coûts. 


## 1. Données

In [ ]:
import sys, time, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import replace

import portfolio_lab as pl
from portfolio_lab.backtest import WalkForwardBacktestConfig, plot_backtest_nav

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# ------------------------------------------------------------------ #
#  USE_SYNTHETIC=True : tourne sans réseau (sandbox/CI).             #
#  En production, laissez False (vraies données yfinance).          #
# ------------------------------------------------------------------ #
USE_SYNTHETIC = False
START, END = "2012-01-01", "2025-12-31"

In [ ]:
# --- Univers CONFIGURABLE (Option 1 : complet ; Option 2 : sous-échantillon) ---
# Décommentez le sélecteur voulu. Tout le reste du notebook est inchangé.
constituents = pl.get_sp500_constituents()

SELECTOR = pl.StratifiedBySector(n_total=80, seed=42)   # diversifié par secteur (défaut)
# SELECTOR = pl.AllUniverse()                            # Option 1 : S&P 500 complet (~500, lent)
# SELECTOR = pl.TopN(40)                                 # N premiers (ALPHABÉTIQUE)
# SELECTOR = pl.RandomN(50, seed=42)                     # aléatoire reproductible

universe = SELECTOR.select(constituents)
if USE_SYNTHETIC:
    prices = pl.make_synthetic_prices(constituents.loc[universe], START, END)
else:
    prices = pl.clean_prices(pl.download_prices(universe, START, END))

returns = pl.compute_returns(prices, "simple")
print(f"Univers : {prices.shape[1]} actifs | {prices.shape[0]} obs "
      f"({prices.index[0].date()} -> {prices.index[-1].date()})")

In [ ]:
# Backtest walk-forward — IDENTIQUE pour toutes les variantes (variables contrôlées).
BT_CFG = WalkForwardBacktestConfig(
    estimation_window=252,      # 1 an pour estimer mu, Sigma
    holding_period=21,          # rebalancement ~mensuel
    transaction_cost_bps=10.0,  # coûts sur le turnover (stratégie ET benchmark 1/N)
    long_only=True,
    frequency="daily",
    risk_free_annual=0.04,      # Sharpe NET du taux sans risque
)
PPY = pl.periods_per_year(BT_CFG.frequency)
WINDOW, STEP = BT_CFG.estimation_window, BT_CFG.holding_period

In [ ]:
# Les 7 estimateurs de covariance comparés 
estimators = {
    "Rolling-120":      pl.RollingCovariance(window=120),
    "EWMA λ=0.94":      pl.EWMACovariance(lam=0.94),
    "EWMA λ=0.97":      pl.EWMACovariance(lam=0.97),
    "IEWMA":            pl.IEWMACovariance(vol_halflife=125, corr_halflife=21),
    "Ledoit-Wolf":      pl.LedoitWolfCovariance(),
    "OAS":              pl.OASCovariance(),
    "Corr. constante":  pl.ConstantCorrelationCovariance(),
}
MEAN_EST = pl.EWMAMean(lam=0.9)
# MEAN_EST = pl.SampleMean()  
print("Estimateurs Σ :")
for name, est in estimators.items():
    print(f"  {name}")

## 2. Evaluation statistique (hors échantillon)

In [ ]:
# Résumé statistique (moyenne sur toutes les fenêtres OOS) par estimateur.
stat = pl.cov_eval_summary(returns, estimators, window=WINDOW, step=STEP, demean=False)
stat["stabilité"] = {name: pl.temporal_stability(returns, est, window=WINDOW, step=STEP)
                     for name, est in estimators.items()}
cols = ["loglik", "rmse_cov", "mae_cov", "frobenius", "vol_abs_err",
        "condition_number", "stabilité"]
display(stat[cols].round({"loglik": 2, "rmse_cov": 6, "mae_cov": 6, "frobenius": 6,
                          "vol_abs_err": 5, "condition_number": 1, "stabilité": 4}))

In [ ]:
# Lecture visuelle : log-vraisemblance (haut = meilleur prédicteur) et conditionnement (bas = mieux).
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
stat["loglik"].plot(kind="bar", ax=axes[0], color="seagreen", edgecolor="black")
axes[0].set_title("Log-vraisemblance OOS (↑ meilleur)")
stat["vol_abs_err"].plot(kind="bar", ax=axes[1], color="slateblue", edgecolor="black")
axes[1].set_title("Erreur de prévision de vol (↓ meilleur)")
stat["condition_number"].plot(kind="bar", ax=axes[2], color="indianred", edgecolor="black")
axes[2].set_title("Conditionnement (↓ meilleur)")
for a in axes: a.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Conditionnement DANS LE TEMPS (stabilité numérique) — quelques estimateurs.
fig, ax = plt.subplots(figsize=(13, 5))
for name in ["Empirique", "EWMA λ=0.94", "IEWMA", "Ledoit-Wolf"]:
    df = pl.rolling_cov_eval(returns, estimators[name], window=WINDOW, step=STEP)
    ax.plot(df.index, df["condition_number"], label=name, lw=1.6)
ax.set_yscale("log"); ax.set_ylabel("Conditionnement (échelle log)")
ax.set_title("Conditionnement de Σ au fil du temps"); ax.legend(); plt.show()

### Comportement en **crise** vs **hors-crise**

On sépare les fenêtres par quartile de volatilité réalisée (crise = quart le plus volatil) et on
compare conditionnement et erreurs. Attendu : les estimateurs dynamiques (EWMA/IEWMA) résistent
mieux en stress sur le plan prédictif, mais leur conditionnement se dégrade.

In [ ]:
regime_rows = {}
for name, est in estimators.items():
    df = pl.rolling_cov_eval(returns, est, window=WINDOW, step=STEP)
    rs = pl.regime_split(df)
    regime_rows[name] = {
        "loglik (calme)": rs.loc["hors_crise", "loglik"],
        "loglik (crise)": rs.loc["crise", "loglik"],
        "cond (calme)":   rs.loc["hors_crise", "condition_number"],
        "cond (crise)":   rs.loc["crise", "condition_number"],
        "vol_err (crise)": rs.loc["crise", "vol_abs_err"],
    }
display(pd.DataFrame(regime_rows).T.round({"loglik (calme)": 2, "loglik (crise)": 2,
        "cond (calme)": 1, "cond (crise)": 1, "vol_err (crise)": 5}))

## 3. Benchmark portefeuille

### a. On fixe l'objectif min_variance et on fait varier l'estimateur de Σ
y est maximal. μ = SampleMean, optimiseur CVXPY, `w_max=10%`, mêmes fenêtres et coûts.

In [ ]:
base_mv = pl.ExperimentConfig(
    universe=SELECTOR, frequency=BT_CFG.frequency, return_mode="simple",
    mean_estimator=MEAN_EST, objective="min_variance",
    optimizer=pl.CVXPYOptimizer(), long_only=True, w_max=0.10, risk_free_annual=0.04,
)
variants_mv = {name: replace(base_mv, cov_estimator=est) for name, est in estimators.items()}

t0 = time.time()
res_mv = pl.run_oos_benchmark(prices, variants_mv, backtest_config=BT_CFG)
print(f"7 backtests min_variance en {time.time()-t0:.1f}s")

cols_perf = ["Total return (%)", "CAGR (%)", "Vol annualized (%)", "Sharpe",
             "Max drawdown (%)", "Tracking error (%)", "Average turnover", "Fallback count"]
m_mv = pl.metrics_table(res_mv)
display(m_mv[cols_perf].round(3))

In [ ]:
# Métriques de queue + stabilité des poids (robustesse).
display(pl.tail_metrics_table(res_mv, periods_per_year_=PPY, rf_annual=0.04).round(3))
display(pl.weight_stability(res_mv).round(4))

In [ ]:
# Courbes de valeur OOS (base 1) + benchmark 1/N (coûts inclus des deux côtés).
fig, ax = plt.subplots(figsize=(13, 6))
plot_backtest_nav(res_mv, ax=ax)
ax.set_title("min_variance — valeur hors échantillon par estimateur de Σ")
plt.show()

### b. Max_sharpe 
Si la dispersion du Sharpe entre estimateurs **rétrécit** vs `min_variance`, c'est que l'erreur
sur μ domine — améliorer Σ aide surtout là où μ n'intervient pas.

In [ ]:
variants_ms = {name: replace(cfg, objective="max_sharpe") for name, cfg in variants_mv.items()}
res_ms = pl.run_oos_benchmark(prices, variants_ms, backtest_config=BT_CFG)
m_ms = pl.metrics_table(res_ms)
display(m_ms[["CAGR (%)", "Vol annualized (%)", "Sharpe", "Max drawdown (%)",
             "Average turnover", "Fallback count"]].round(3))
print(f"Dispersion Sharpe min_variance : {m_mv['Sharpe'].max()-m_mv['Sharpe'].min():.3f}")
print(f"Dispersion Sharpe max_sharpe   : {m_ms['Sharpe'].max()-m_ms['Sharpe'].min():.3f}")

In [ ]:
# COMPARATIF DES COVARIAANCES POUR CHAQUE PORTFOLIO
base_pf0 = replace(base_mv, objective="max_sharpe", optimizer=pl.CVXPYOptimizer())

In [ ]:
# COMPARATIFS DES COVARIANCES POU EQUIPONDERE 1/N
base_pf = replace(base_pf0, optimizer=pl.EqualWeightOptimizer())
portfolios = {
    "Ledoit-Wolf":      replace(base_pf, cov_estimator=pl.LedoitWolfCovariance()),
    "EWMA":         replace(base_pf, cov_estimator=pl.EWMACovariance(lam=0.94)),
    "Empirique":        replace(base_pf, cov_estimator=pl.EmpiricalCovariance()),
    "Oas":              replace(base_pf, cov_estimator=pl.OASCovariance()),
    "Cor":              replace(base_pf, cov_estimator=pl.ConstantCorrelationCovariance()),
    "IEWMA":            replace(base_pf, cov_estimator=pl.IEWMACovariance(vol_halflife=125, corr_halflife=21)),
    "Rolling":          replace(base_pf, cov_estimator=pl.RollingCovariance(window=120)),
}
res_pf = pl.run_oos_benchmark(prices, portfolios, backtest_config=BT_CFG)
m_pf = pl.metrics_table(res_pf)
display(m_pf[cols_perf].round(3))
display(pl.tail_metrics_table(res_pf, periods_per_year_=PPY, rf_annual=0.04).round(3))

In [ ]:
# COMPARAISONS SUR RISK PARITY
base_pf = replace(base_pf0, optimizer=pl.RiskParityOptimizer())
portfolios = {
    "Ledoit-Wolf":      replace(base_pf, cov_estimator=pl.LedoitWolfCovariance()),
    "EWMA":         replace(base_pf, cov_estimator=pl.EWMACovariance(lam=0.94)),
    "Empirique":        replace(base_pf, cov_estimator=pl.EmpiricalCovariance()),
    "Oas":              replace(base_pf, cov_estimator=pl.OASCovariance()),
    "Cor":              replace(base_pf, cov_estimator=pl.ConstantCorrelationCovariance()),
    "IEWMA":            replace(base_pf, cov_estimator=pl.IEWMACovariance(vol_halflife=125, corr_halflife=21)),
    "Rolling":          replace(base_pf, cov_estimator=pl.RollingCovariance(window=120)),
}
res_pf = pl.run_oos_benchmark(prices, portfolios, backtest_config=BT_CFG)
m_pf = pl.metrics_table(res_pf)
display(m_pf[cols_perf].round(3))
display(pl.tail_metrics_table(res_pf, periods_per_year_=PPY, rf_annual=0.04).round(3))

In [ ]:
# COMPAISON SUR MAX DIVERSIFICATION
base_pf = replace(base_pf0, optimizer=pl.MaxDiversificationOptimizer())
portfolios = {
    "Ledoit-Wolf":      replace(base_pf, cov_estimator=pl.LedoitWolfCovariance()),
    "EWMA":         replace(base_pf, cov_estimator=pl.EWMACovariance(lam=0.94)),
    "Empirique":        replace(base_pf, cov_estimator=pl.EmpiricalCovariance()),
    "Oas":              replace(base_pf, cov_estimator=pl.OASCovariance()),
    "Cor":              replace(base_pf, cov_estimator=pl.ConstantCorrelationCovariance()),
    "IEWMA":            replace(base_pf, cov_estimator=pl.IEWMACovariance(vol_halflife=125, corr_halflife=21)),
    "Rolling":          replace(base_pf, cov_estimator=pl.RollingCovariance(window=120)),
}
res_pf = pl.run_oos_benchmark(prices, portfolios, backtest_config=BT_CFG)
m_pf = pl.metrics_table(res_pf)
display(m_pf[cols_perf].round(3))
display(pl.tail_metrics_table(res_pf, periods_per_year_=PPY, rf_annual=0.04).round(3))

In [ ]:
# Comparison sur MEAN-VARIANCE (max_sharpe)
base_pf = replace(base_pf0, optimizer=pl.CVXPYOptimizer())
portfolios = {
    "Ledoit-Wolf":      replace(base_pf, cov_estimator=pl.LedoitWolfCovariance()),
    "EWMA":         replace(base_pf, cov_estimator=pl.EWMACovariance(lam=0.94)),
    "Empirique":        replace(base_pf, cov_estimator=pl.EmpiricalCovariance()),
    "Oas":              replace(base_pf, cov_estimator=pl.OASCovariance()),
    "Cor":              replace(base_pf, cov_estimator=pl.ConstantCorrelationCovariance()),
    "IEWMA":            replace(base_pf, cov_estimator=pl.IEWMACovariance(vol_halflife=125, corr_halflife=21)),
    "Rolling":          replace(base_pf, cov_estimator=pl.RollingCovariance(window=120)),
}
res_pf = pl.run_oos_benchmark(prices, portfolios, backtest_config=BT_CFG)
m_pf = pl.metrics_table(res_pf)
display(m_pf[cols_perf].round(3))
display(pl.tail_metrics_table(res_pf, periods_per_year_=PPY, rf_annual=0.04).round(3))

### On fixe le Σ (Ledoit-Wolf) et on fait varier le type de portefeuille
Cinq constructions : Équipondéré 1/N, Min variance, Risk parity, Max diversification,
Mean-Variance (max Sharpe). Isole l'effet de la *méthode d'allocation* à Σ donné.

In [ ]:
# Comparatif de plusieurs stratégies de portefeuille, toutes avec Σ = Ledoit-Wolf 
base_pf = replace(base_mv, cov_estimator=pl.LedoitWolfCovariance())
portfolios = {
    "Équipondéré 1/N":      replace(base_pf, optimizer=pl.EqualWeightOptimizer()),
    "Min variance":         replace(base_pf, objective="min_variance", optimizer=pl.CVXPYOptimizer()),
    "Risk parity":          replace(base_pf, optimizer=pl.RiskParityOptimizer()),
    "Max diversification":  replace(base_pf, optimizer=pl.MaxDiversificationOptimizer()),
    "Mean-Variance (Sharpe)": replace(base_pf, objective="max_sharpe", optimizer=pl.CVXPYOptimizer()),
}
res_pf = pl.run_oos_benchmark(prices, portfolios, backtest_config=BT_CFG)
m_pf = pl.metrics_table(res_pf)
display(m_pf[cols_perf].round(3))
display(pl.tail_metrics_table(res_pf, periods_per_year_=PPY, rf_annual=0.04).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
plot_backtest_nav(res_pf, ax=ax)
ax.set_title("Types de portefeuille (Σ = Ledoit-Wolf) — valeur hors échantillon")
plt.show()

## 4. Exemple concret d'allocation de 100 000 € 

On estime/optimise UNIQUEMENT sur le passé (≤ split), on fige les poids, on les applique au futur, et on compare au 1/N (coûts de friction inclus côté backtest).

In [ ]:
CAPITAL, SPLIT = 100_000.0, "2023-12-31"
best_stat = stat["loglik"].idxmax()          
print(f"Estimateur retenu (meilleure log-vraisemblance) : {best_stat}")
best_cfg = replace(base_mv, cov_estimator=estimators[best_stat])

ex = pl.train_test_split_evaluation(prices, best_cfg, split_date=SPLIT, capital=CAPITAL,
                                    frequency=BT_CFG.frequency, rf_annual=0.04)
print(f"Passé : {ex['n_past']} obs | Futur : {ex['n_future']} obs | Split : {ex['split_date']}")
display(ex["allocation"].head(12))
display(ex["comparison"].round(2))

fig, ax = plt.subplots(figsize=(13, 6))
ex["strategy_nav"].plot(ax=ax, lw=2, label=f"min_variance / {best_stat}")
ex["benchmark_nav"].plot(ax=ax, lw=2, ls="--", color="black", label="Équipondéré 1/N")
ax.axhline(CAPITAL, color="gray", lw=1, alpha=0.6)
ax.set_title(f"Valeur sur le futur — capital initial {CAPITAL:,.0f} €")
ax.set_ylabel("Valeur (€)"); ax.legend(); plt.show()

## Recommandations finales

On agrège les deux axes pour classer les estimateurs. 

In [ ]:
ranking = pd.DataFrame({
    "stat: loglik (↑)":      stat["loglik"].rank(ascending=False),
    "stat: vol_err (↓)":     stat["vol_abs_err"].rank(ascending=True),
    "stat: cond. (↓)":       stat["condition_number"].rank(ascending=True),
    "stat: stabilité (↓)":   stat["stabilité"].rank(ascending=True),
    "pf: Sharpe minVar (↑)": m_mv["Sharpe"].rank(ascending=False),
    "pf: vol minVar (↓)":    m_mv["Vol annualized (%)"].rank(ascending=True),
    "pf: turnover (↓)":      pl.weight_stability(res_mv)["turnover moyen"].rank(ascending=True),
})
ranking["RANG MOYEN"] = ranking.mean(axis=1)
display(ranking.sort_values("RANG MOYEN").round(2))

print("\nMeilleur prédictif (loglik)        :", stat["loglik"].idxmax())
print("Meilleur risque (vol minVar OOS)   :", m_mv["Vol annualized (%)"].idxmin())
print("Meilleur conditionnement           :", stat["condition_number"].idxmin())
print("Meilleur Sharpe portefeuille (minVar):", m_mv["Sharpe"].idxmax())
print("Compromis global (rang moyen)      :", ranking["RANG MOYEN"].idxmin())

##  Journalisation du run (`results/`)




In [ ]:
from portfolio_lab.reporting import RunReport

rep = RunReport("covariance", results_dir="results")
rep.capture(
    prices=prices, selector=SELECTOR,
    requested_n=getattr(SELECTOR, "n_total", getattr(SELECTOR, "n", None)),
    period=(START, END), bt_cfg=BT_CFG, base_config=base_mv, use_synthetic=USE_SYNTHETIC,
    cov_estimators=estimators, mean_estimators={"μ": MEAN_EST},
)
ns = globals()
rep.add_if(ns, "stat",        "Qualité statistique de Σ (OOS)")
rep.add_if(ns, "regime_rows", "Régime crise / hors-crise")
rep.add_if(ns, "res_mv",      "Portefeuille min_variance — on varie Σ", columns=cols_perf)
rep.add_if(ns, "res_ms",      "Portefeuille max_sharpe — on varie Σ")
print("Rapport écrit :", rep.save())   # rep.save(fmt="both") pour tenter un PDF (pandoc)
